In [1]:
import os

%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\Chicken-Disease-Classification\\notebooks'

In [2]:
os.chdir("../")
%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\Chicken-Disease-Classification'

In [3]:
from dataclasses import dataclass
from pathlib import Path
import torch

model = torch.load("artifacts/training/model.pth")

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    params_image_size: list
    params_batch_size: int

In [4]:
from chicken_disease_classification.constants import *
from chicken_disease_classification.utils.common import read_yaml, create_directories

class ConfigManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_evaluation_config(self) -> EvaluationConfig:
        return EvaluationConfig(
            path_of_model=Path(self.config.prepare_callbacks_config.best_model_path),
            training_data=Path(self.config.data_ingestion_config.unzipped_data_dir),
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )

In [11]:
import torch
import torch.nn as nn
from torchvision import models, datasets
from torch.utils.data import DataLoader, Subset
from pathlib import Path

from chicken_disease_classification.utils.common import save_json
from chicken_disease_classification.utils.dataloader import get_base_aug

class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self._val_loader()
    
    def _val_loader(self):
        transform = get_base_aug(config=self.config)
        full_dataset = datasets.ImageFolder(root=self.config.training_data, transform=transform)
        val_size = int(0.2 * len(full_dataset))
        indices = list(range(len(full_dataset)))

        self.val_loader = DataLoader(
            Subset(full_dataset, indices[-val_size:]),
            batch_size=self.config.params_batch_size,
            shuffle=False
        )
        self.num_classes = len(full_dataset.classes)

    def load_model(self):
        self.model = models.vgg16()
        self.model.classifier[6] = nn.Linear(
            self.model.classifier[6].in_features,
            self.num_classes
        )
        self.model.load_state_dict(
            torch.load(self.config.path_of_model, map_location=self.device)
        )
        self.model = self.model.to(self.device)
        self.model.eval()
    
    @torch.no_grad()
    def evaluation(self):
        criterion = nn.CrossEntropyLoss()
        total_loss, correct, total = 0.0, 0, 0

        for images, labels in self.val_loader:
            images, labels = images.to(self.device), labels.to(self.device)
            outputs = self.model(images)
            total_loss += criterion(outputs, labels).item()
            correct += outputs.max(1)[1].eq(labels).sum().item()
            total += labels.size(0)

        self.score = {
            "loss": total_loss / len(self.val_loader),
            "accuracy": correct / total
        }

    def save_score(self):
        save_json(path=Path("scores.json"), content=self.score)

In [12]:
import sys
from chicken_disease_classification.exception.exception import CustomException

try:
    config = ConfigManager()
    val_config = config.get_evaluation_config()
    evaluation = Evaluation(val_config)
    evaluation.load_model()
    evaluation.evaluation()
    evaluation.save_score()
except Exception as e:
   raise CustomException(e, sys)

[2026-03-31 16:15:29,366: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-31 16:15:29,368: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-31 16:15:29,370: INFO: common: created directory at: artifacts]
